# Video Metrics

Review temporal video model metrics and artifacts.

Steps:
- Load available vision/video metrics.
- Inspect score files.
- Summarize plots and artifacts.



In [ ]:
from __future__ import annotations

import json
import os
import sys
import subprocess
from pathlib import Path

# Resolve repo root from the notebook location.
REPO_ROOT = Path.cwd()
for parent in [REPO_ROOT] + list(REPO_ROOT.parents):
    if (parent / 'scripts').exists() and (parent / 'notebooks').exists():
        REPO_ROOT = parent
        break

# Ensure local modules are importable.
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / 'src'))

PY = sys.executable

def run(cmd: list[str]) -> None:
    # Run a command from the repo root with PYTHONPATH set.
    env = os.environ.copy()
    env['PYTHONPATH'] = os.pathsep.join([str(REPO_ROOT / 'src'), str(REPO_ROOT)])
    print('$', ' '.join(cmd))
    subprocess.run(cmd, cwd=str(REPO_ROOT), check=True, env=env)

def show_json(rel_path: str) -> None:
    path = REPO_ROOT / rel_path
    if not path.exists():
        print('Missing:', path)
        return
    try:
        data = json.loads(path.read_text(encoding='utf-8'))
    except Exception:
        print(path.read_text(encoding='utf-8', errors='ignore')[:2000])
        return
    print(json.dumps(data, indent=2))

def list_dir(rel_path: str, limit: int = 20) -> None:
    path = REPO_ROOT / rel_path
    if not path.exists():
        print('Missing:', path)
        return
    print(f'\n{rel_path}/')
    for item in sorted(path.iterdir())[:limit]:
        print(' -', item.name)


In [ ]:
import pandas as pd
from pathlib import Path

summary = {
    'metrics': {},
    'scores': {},
    'plots': [],
}

metrics_paths = [
    REPO_ROOT / 'experiments' / 'vision' / 'video_temporal' / 'metrics.json',
    REPO_ROOT / 'experiments' / 'vision' / 'temporal_lstm' / 'metrics.json',
    REPO_ROOT / 'experiments' / 'vision' / 'metrics.csv',
]

for path in metrics_paths:
    if not path.exists():
        continue
    if path.suffix == '.json':
        data = json.loads(path.read_text(encoding='utf-8'))
        summary['metrics'][str(path.relative_to(REPO_ROOT))] = data
        print(path.name, data)
    else:
        df = pd.read_csv(path)
        summary['metrics'][str(path.relative_to(REPO_ROOT))] = {
            'rows': int(df.shape[0]),
            'cols': int(df.shape[1]),
            'columns': list(df.columns),
        }
        print('Metrics CSV:', path.name, df.shape)


In [ ]:
# Inspect video score files.
scores_path = REPO_ROOT / 'experiments' / 'vision' / 'scores.csv'
if scores_path.exists():
    df = pd.read_csv(scores_path)
    summary['scores']['rows'] = int(df.shape[0])
    summary['scores']['columns'] = list(df.columns)
    print('Scores shape:', df.shape)
    print(df.head(10))
else:
    print('Missing:', scores_path)


In [ ]:
# List plot artifacts if available.
plots_dir = REPO_ROOT / 'experiments' / 'vision' / 'plots'
if plots_dir.exists():
    plot_files = [str(p.relative_to(REPO_ROOT)) for p in plots_dir.iterdir() if p.is_file()]
    summary['plots'] = plot_files
    for item in plot_files:
        print(' -', item)
else:
    print('Missing:', plots_dir)


In [ ]:
# Persist summary for quick reference.
report_dir = REPO_ROOT / 'reports'
report_dir.mkdir(parents=True, exist_ok=True)
summary_path = report_dir / 'evaluation_video_metrics_summary.json'
summary_path.write_text(json.dumps(summary, indent=2))
print('Saved summary to', summary_path)


In [ ]:
# Quick artifact index for verification.
for folder in ['models', 'experiments', 'artifacts', 'runs', 'reports', 'logs']:
    path = REPO_ROOT / folder
    if not path.exists():
        continue
    print(f'\n{folder}/')
    for item in sorted(path.iterdir())[:20]:
        print(' -', item.name)
